In [1]:
# Cell 1 - imports
import os
import gc
import random
import tqdm
from collections import defaultdict

import numpy as np
import torch

In [6]:
# Cell 2 - configuration + useful functions
#
# Goal: turn each "<bench>_homogeneous" performance dataset into a
# "<bench>_categoriesbalanced" one by partitioning whole CATEGORIES into
# train/test so that
#   (1) train holds ~90% of the datapoints (test ~10%),
#   (2) train and test have similar accuracy, for ALL models at once.
#
# Loss for a candidate split (both terms are fractions in [0, 1], so a single
# shared constant WEIGHT balances them):
#   loss = |frac_train - 0.90|  +  WEIGHT * mean_over_models|acc_train - acc_test|
# The accuracy term averages every model's train/test accuracy gap, so the
# chosen split is good for all models, not just one.

DATA_PATH = "data/datasets/processed_with_performance"   # same location as notebook 1
VERSION = "quantized"                                # all data are quantized now
SPLITS = ["train", "test"]

MODELS = [
    "Qwen3.5-9B",
    "Llama-3.2-3B-Instruct",
    "phi-4",
    "Mistral-7B-Instruct-v0.3",
    "Llama-2-7b-chat",
    "Qwen2.5-7B-Instruct",
    #"gemma-3-12b-it",
    #"gemma-4-31B-it",
    #"Falcon-H1R-7B",
    #"gemma-3-4b-it",
    "Qwen3.5-9B-Base",
    "Qwen2.5-7B",
    #"gemma-3-4b-pt",
    "Ministral-3-3B-Instruct-2512",
    "Ministral-3-8B-Instruct-2512",
    #"Falcon-H1-7B-Instruct"
]

# Datasets from make_perf_all.sh. The trailing number is the number of MCQ
# options (math10 -> 10 options, math4 -> 4, ...), but we never rely on the
# name for that: nb_options is read per question from the output length.
DATASETS = [
    'math10_homogeneous',
    'math4_homogeneous',
    'mmlupro4_homogeneous',
    'mmlupro10_homogeneous',
    'medmcqa4_homogeneous',
]

TARGET_TRAIN = 0.90      # desired train fraction of datapoints
N_TRIALS = 10_000        # random splits tried per benchmark
SEED = 42                # reproducibility
WEIGHT = 1.0             # same constant on both loss terms (both are %)


def file_path(dataset, model):
    return os.path.join(DATA_PATH, dataset, f"{model}_{VERSION}_dataset")


def cb_name(dataset):
    """math10_homogeneous -> math10_categoriesbalanced (next to the original)."""
    if "_homogeneous" in dataset:
        return dataset.replace("_homogeneous", "_categoriesbalanced")
    return dataset + "_categoriesbalanced"


def _torch_load(path):
    try:
        return torch.load(path, weights_only=False)   # files hold tensors + lists
    except TypeError:
        return torch.load(path)                        # older torch


def category_stats_for_dataset(dataset):
    """Per-category counts + per-model per-category accuracy sums.

    RAM-light: loads ONE model file at a time, keeps only the scalar
    aggregates (categories + accuracies), and frees the heavy tensors before
    moving on. The big `outputs`/`logits` are never accumulated here.
    """
    per_model_sumacc, per_model_count, models_found = {}, {}, []
    for model in MODELS:
        path = file_path(dataset, model)
        if not os.path.exists(path):
            print(f"  [skip] missing file: {path}")
            continue
        data = _torch_load(path)
        sumacc, cnt = defaultdict(float), defaultdict(int)
        has_cats = True
        for split in SPLITS:
            if split not in data:
                continue
            meta = data[split]
            if "categories" not in meta:
                has_cats = False
                break
            for c, a in zip(meta["categories"], meta["accuracies"]):
                sumacc[c] += float(a)   # float() is robust to old tensor accuracies
                cnt[c] += 1
        del data
        gc.collect()
        if not has_cats:
            print(f"  [skip] {dataset}: no 'categories' field -> cannot rebalance")
            return None
        per_model_sumacc[model] = dict(sumacc)
        per_model_count[model] = dict(cnt)
        models_found.append(model)

    if not models_found:
        print(f"  [skip] {dataset}: no model files found")
        return None

    # Category counts are the same questions for every model.
    counts = per_model_count[models_found[0]]
    for m in models_found[1:]:
        if per_model_count[m] != counts:
            print(counts)
            print(per_model_count[m])
            print(f"  [warn] {dataset}: category counts differ for {m} (using first model)")
    categories = sorted(counts)
    if len(categories) < 2:
        print(f"  [skip] {dataset}: needs >=2 categories to split")
        return None
    total = sum(counts.values())
    return dict(dataset=dataset, categories=categories, counts=counts, total=total,
                per_model_sumacc=per_model_sumacc, per_model_count=per_model_count,
                models_found=models_found)


def evaluate_split(test_cats, stats, weight=WEIGHT):
    """Return loss + breakdown for putting `test_cats` in test, rest in train."""
    test_cats = set(test_cats)
    categories, counts, total = stats["categories"], stats["counts"], stats["total"]
    train_cats = [c for c in categories if c not in test_cats]
    if not train_cats or not test_cats:
        return None
    count_test = sum(counts[c] for c in test_cats)
    frac_train = (total - count_test) / total
    size_term = abs(frac_train - TARGET_TRAIN)

    diffs, accs = [], {}
    for m in stats["models_found"]:
        sa, cn = stats["per_model_sumacc"][m], stats["per_model_count"][m]
        tr_n = sum(cn.get(c, 0) for c in train_cats)
        tr_s = sum(sa.get(c, 0.0) for c in train_cats)
        te_n = sum(cn.get(c, 0) for c in test_cats)
        te_s = sum(sa.get(c, 0.0) for c in test_cats)
        if tr_n == 0 or te_n == 0:
            return None
        at, ae = tr_s / tr_n, te_s / te_n
        diffs.append(abs(at - ae))
        accs[m] = (at, ae)
    acc_term = sum(diffs) / len(diffs)

    return dict(loss=size_term + weight * acc_term,
                size_term=size_term, acc_term=acc_term, frac_train=frac_train,
                train_cats=sorted(train_cats), test_cats=sorted(test_cats), accs=accs)


def _random_test_categories(categories, counts, total, rng, low=0.05, high=0.15):
    """Sample a candidate test set: shuffle categories, greedily add whole
    categories until their cumulative datapoint fraction reaches a random
    target near 10%. Keeps every trial close to the 90/10 region instead of
    wasting trials on wildly unbalanced splits."""
    target = rng.uniform(low, high)
    order = list(categories)
    rng.shuffle(order)
    test, frac = [], 0.0
    for c in order:
        if frac >= target:
            break
        test.append(c)
        frac += counts[c] / total
    return test


def search_best_split(stats, n_trials=N_TRIALS, seed=SEED, weight=WEIGHT):
    """Random search: try `n_trials` candidate splits, keep the lowest loss.
    A fresh RNG seeded with `seed` makes the result reproducible and
    independent of dataset ordering."""
    rng = random.Random(seed)
    best = None
    for _ in tqdm.tqdm(range(n_trials)):
        cand = _random_test_categories(stats["categories"], stats["counts"],
                                       stats["total"], rng)
        res = evaluate_split(cand, stats, weight=weight)
        if res is None:
            continue
        if best is None or res["loss"] < best["loss"]:
            best = res
    return best


def reassign_and_save(dataset, train_cats, test_cats, dry_run=False):
    """Rebuild every model's file with categories repartitioned into the new
    train/test, saving under <bench>_categoriesbalanced.

    RAM-careful: processes ONE model at a time; the new train/test lists store
    references to the SAME tensor objects (no copying of outputs/logits), and
    the source object is freed before saving.
    """
    train_cats = set(train_cats)
    out_dir = os.path.join(DATA_PATH, cb_name(dataset))
    os.makedirs(out_dir, exist_ok=True)

    try:
        from datasets import DatasetDict
        wrap = lambda tr, te: DatasetDict({"train": tr, "test": te})
    except Exception:
        wrap = lambda tr, te: {"train": tr, "test": te}

    for model in MODELS:
        src = file_path(dataset, model)
        if not os.path.exists(src):
            print(f"  [skip] missing file: {src}")
            continue
        data = _torch_load(src)
        fields = list(data["train"].keys())
        new_train = {f: [] for f in fields}
        new_test = {f: [] for f in fields}
        for split in SPLITS:
            if split not in data:
                continue
            meta = data[split]
            keep = [c in train_cats for c in meta["categories"]]
            for f in fields:
                col = meta[f]
                new_train[f].extend(v for v, k in zip(col, keep) if k)
                new_test[f].extend(v for v, k in zip(col, keep) if not k)
        del data
        gc.collect()

        n_tr = len(new_train["questions"])
        n_te = len(new_test["questions"])
        dst = os.path.join(out_dir, f"{model}_{VERSION}_dataset")
        if not dry_run:
            torch.save(wrap(new_train, new_test), dst)
        del new_train, new_test
        gc.collect()
        print(f"  {model}: train={n_tr:,} test={n_te:,} -> {dst}"
              + ("  [dry run]" if dry_run else ""))

In [8]:
# Cell 3 - compute the best split per benchmark (over all models) and report it
import statistics as _stats


def _split_report(name, stats, res, note=None):
    """Print the per-model table plus mean/delta rows for a split result."""
    total   = stats["total"]
    n_test  = sum(stats["counts"][c] for c in res["test_cats"])
    n_train = total - n_test
    print("=" * 78)
    print(name + (f"   [{note}]" if note else ""))
    print(f"  models in loss : {stats['models_found']}")
    print(f"  train categories ({len(res['train_cats'])}): {res['train_cats']}")
    print(f"  test  categories ({len(res['test_cats'])}): {res['test_cats']}")
    print(f"  datapoints     : train {n_train:,} ({res['frac_train']:.1%}) | "
          f"test {n_test:,} ({n_test / total:.1%})   target {TARGET_TRAIN:.0%} / "
          f"{1 - TARGET_TRAIN:.0%}")
    print(f"  loss = {res['loss']:.4f}  (size_term={res['size_term']:.4f}, "
          f"acc_term={res['acc_term']:.4f})")
    print(f"  {'model':<26} {'train_acc':>10} {'test_acc':>10} {'|diff|':>8}")
    for m in stats["models_found"]:
        at, ae = res["accs"][m]
        print(f"  {m:<26} {at:>10.4f} {ae:>10.4f} {abs(at - ae):>8.4f}")

    at_all = [res["accs"][m][0] for m in stats["models_found"]]
    ae_all = [res["accs"][m][1] for m in stats["models_found"]]
    mt, me = _stats.mean(at_all), _stats.mean(ae_all)
    print(f"  {'-' * 56}")
    print(f"  {f'MEAN (n={len(at_all)})':<26} {mt:>10.4f} {me:>10.4f} "
          f"{abs(mt - me):>8.4f}")
    print(f"  {'delta (train - test)':<26} {mt - me:>+10.4f}"
          f"   mean|diff| = {res['acc_term']:.4f}")
    return res


# ------------------------------------------------------------ main search ---
splits_by_dataset, stats_by_dataset = {}, {}
for dataset in DATASETS:
    stats = category_stats_for_dataset(dataset)
    if stats is None:
        print("=" * 78); print(f"{dataset}\n  [skip] no stats")
        continue
    stats_by_dataset[dataset] = stats
    best = search_best_split(stats)
    if best is None:
        print("=" * 78); print(f"{dataset}\n  [skip] no valid split found")
        continue
    splits_by_dataset[dataset] = (best["train_cats"], best["test_cats"])
    _split_report(dataset, stats, best)

print("=" * 78)
print(f"computed splits for {len(splits_by_dataset)} / {len(DATASETS)} datasets")


# ------------------------------------------- cross-applied splits (checks) ---
# Reuse the split searched on <source> to evaluate <target>, so the two variants
# differ only in the number of options, not in the split.
CROSS_SPLITS = [("mmlupro4_homogeneous", "mmlupro10_homogeneous")]

for target, source in CROSS_SPLITS:
    if target not in stats_by_dataset or source not in splits_by_dataset:
        print(f"[skip cross] {target} <- {source}: missing stats or split")
        continue
    st = stats_by_dataset[target]
    _, src_test = splits_by_dataset[source]

    missing = set(src_test) - set(st["categories"])
    if missing:
        print(f"[skip cross] {target} <- {source}: categories absent from target: "
              f"{sorted(missing)}")
        continue
    extra = set(st["categories"]) - set(stats_by_dataset[source]["categories"])
    if extra:
        print(f"  [warn] {target} has categories {source} lacks (-> train): "
              f"{sorted(extra)}")

    res = evaluate_split(src_test, st)
    if res is None:
        print(f"[skip cross] {target} <- {source}: evaluate_split returned None")
        continue

    own_test = splits_by_dataset[target][1]
    _split_report(target, st, res, note=f"split reused from {source}")
    same = sorted(own_test) == sorted(res["test_cats"])
    print(f"  identical to own searched split: {same}")
    if not same:
        own = evaluate_split(own_test, st)
        print(f"  loss: own={own['loss']:.4f}  reused={res['loss']:.4f}  "
              f"(delta {res['loss'] - own['loss']:+.4f})")

100%|##########| 10000/10000 [00:00<00:00, 22040.46it/s]


math10_homogeneous
  models in loss : ['Qwen3.5-9B', 'Llama-3.2-3B-Instruct', 'phi-4', 'Mistral-7B-Instruct-v0.3', 'Llama-2-7b-chat', 'Qwen2.5-7B-Instruct', 'Qwen3.5-9B-Base', 'Qwen2.5-7B', 'Ministral-3-3B-Instruct-2512', 'Ministral-3-8B-Instruct-2512']
  train categories (8): ['easy+', 'easy-', 'hard*', 'hard+', 'hard-', 'medium*', 'medium+', 'medium-']
  test  categories (1): ['easy*']
  datapoints     : train 108,000 (90.0%) | test 12,000 (10.0%)   target 90% / 10%
  loss = 0.0329  (size_term=0.0000, acc_term=0.0329)
  model                       train_acc   test_acc   |diff|
  Qwen3.5-9B                     0.2914     0.2182   0.0732
  Llama-3.2-3B-Instruct          0.1372     0.1278   0.0095
  phi-4                          0.2550     0.2841   0.0291
  Mistral-7B-Instruct-v0.3       0.1275     0.1353   0.0079
  Llama-2-7b-chat                0.1626     0.1921   0.0295
  Qwen2.5-7B-Instruct            0.2974     0.2983   0.0009
  Qwen3.5-9B-Base                0.2997     0.2262   0

100%|##########| 10000/10000 [00:00<00:00, 29873.49it/s]


math4_homogeneous
  models in loss : ['Qwen3.5-9B', 'Llama-3.2-3B-Instruct', 'phi-4', 'Mistral-7B-Instruct-v0.3', 'Llama-2-7b-chat', 'Qwen2.5-7B-Instruct', 'Qwen3.5-9B-Base', 'Qwen2.5-7B', 'Ministral-3-3B-Instruct-2512', 'Ministral-3-8B-Instruct-2512']
  train categories (8): ['easy+', 'easy-', 'hard*', 'hard+', 'hard-', 'medium*', 'medium+', 'medium-']
  test  categories (1): ['easy*']
  datapoints     : train 108,000 (90.0%) | test 12,000 (10.0%)   target 90% / 10%
  loss = 0.0438  (size_term=0.0000, acc_term=0.0438)
  model                       train_acc   test_acc   |diff|
  Qwen3.5-9B                     0.5585     0.4494   0.1090
  Llama-3.2-3B-Instruct          0.3407     0.2863   0.0544
  phi-4                          0.5269     0.5288   0.0019
  Mistral-7B-Instruct-v0.3       0.3122     0.2685   0.0437
  Llama-2-7b-chat                0.3177     0.3224   0.0048
  Qwen2.5-7B-Instruct            0.5225     0.5014   0.0211
  Qwen3.5-9B-Base                0.5413     0.4390   0.

100%|##########| 10000/10000 [00:00<00:00, 23705.57it/s]


mmlupro4_homogeneous
  models in loss : ['Qwen3.5-9B', 'Llama-3.2-3B-Instruct', 'phi-4', 'Mistral-7B-Instruct-v0.3', 'Llama-2-7b-chat', 'Qwen2.5-7B-Instruct', 'Qwen3.5-9B-Base', 'Qwen2.5-7B', 'Ministral-3-3B-Instruct-2512', 'Ministral-3-8B-Instruct-2512']
  train categories (12): ['biology', 'business', 'chemistry', 'economics', 'engineering', 'health', 'history', 'law', 'math', 'philosophy', 'physics', 'psychology']
  test  categories (2): ['computer science', 'other']
  datapoints     : train 10,611 (88.9%) | test 1,330 (11.1%)   target 90% / 10%
  loss = 0.0361  (size_term=0.0114, acc_term=0.0247)
  model                       train_acc   test_acc   |diff|
  Qwen3.5-9B                     0.5227     0.5532   0.0305
  Llama-3.2-3B-Instruct          0.4184     0.4349   0.0165
  phi-4                          0.6142     0.6494   0.0352
  Mistral-7B-Instruct-v0.3       0.4597     0.4882   0.0285
  Llama-2-7b-chat                0.3519     0.3540   0.0021
  Qwen2.5-7B-Instruct           

100%|##########| 10000/10000 [00:00<00:00, 23818.70it/s]


mmlupro10_homogeneous
  models in loss : ['Qwen3.5-9B', 'Llama-3.2-3B-Instruct', 'phi-4', 'Mistral-7B-Instruct-v0.3', 'Llama-2-7b-chat', 'Qwen2.5-7B-Instruct', 'Qwen3.5-9B-Base', 'Qwen2.5-7B', 'Ministral-3-3B-Instruct-2512', 'Ministral-3-8B-Instruct-2512']
  train categories (11): ['biology', 'business', 'chemistry', 'economics', 'engineering', 'health', 'law', 'math', 'other', 'physics', 'psychology']
  test  categories (3): ['computer science', 'history', 'philosophy']
  datapoints     : train 10,337 (89.6%) | test 1,202 (10.4%)   target 90% / 10%
  loss = 0.0330  (size_term=0.0042, acc_term=0.0289)
  model                       train_acc   test_acc   |diff|
  Qwen3.5-9B                     0.3391     0.3857   0.0466
  Llama-3.2-3B-Instruct          0.2563     0.2545   0.0019
  phi-4                          0.4663     0.5408   0.0745
  Mistral-7B-Instruct-v0.3       0.2826     0.2899   0.0073
  Llama-2-7b-chat                0.1832     0.1732   0.0100
  Qwen2.5-7B-Instruct          

100%|##########| 10000/10000 [00:00<00:00, 18328.10it/s]


medmcqa4_homogeneous
  models in loss : ['Qwen3.5-9B', 'Llama-3.2-3B-Instruct', 'phi-4', 'Mistral-7B-Instruct-v0.3', 'Llama-2-7b-chat', 'Qwen2.5-7B-Instruct', 'Qwen3.5-9B-Base', 'Qwen2.5-7B', 'Ministral-3-3B-Instruct-2512', 'Ministral-3-8B-Instruct-2512']
  train categories (18): ['Anaesthesia', 'Anatomy', 'Biochemistry', 'Dental', 'ENT', 'Forensic Medicine', 'Gynaecology & Obstetrics', 'Medicine', 'Microbiology', 'Orthopaedics', 'Pathology', 'Pediatrics', 'Pharmacology', 'Psychiatry', 'Radiology', 'Social & Preventive Medicine', 'Surgery', 'Unknown']
  test  categories (3): ['Ophthalmology', 'Physiology', 'Skin']
  datapoints     : train 108,642 (90.0%) | test 12,123 (10.0%)   target 90% / 10%
  loss = 0.0077  (size_term=0.0004, acc_term=0.0073)
  model                       train_acc   test_acc   |diff|
  Qwen3.5-9B                     0.5699     0.5657   0.0041
  Llama-3.2-3B-Instruct          0.6686     0.6556   0.0130
  phi-4                          0.6874     0.6855   0.0019
  M

In [ ]:
# Cell 4 - save the rebalanced datasets for every model and benchmark
# Set DRY_RUN = True to preview sizes/paths without writing anything.
DRY_RUN = True

if not splits_by_dataset:
    print("No splits computed (run Cell 3 first).")
else:
    for dataset, (train_cats, test_cats) in splits_by_dataset.items():
        print("=" * 78)
        print(f"{dataset} -> {cb_name(dataset)}")
        reassign_and_save(dataset, train_cats, test_cats, dry_run=DRY_RUN)
    print("=" * 78)
    print("done" + (" (dry run)" if DRY_RUN else ""))

math10_homogeneous -> math10_categoriesbalanced
  Qwen3.5-9B: train=108,000 test=12,000 -> datasets/processed_with_performance/math10_categoriesbalanced/Qwen3.5-9B_quantized_dataset
  Llama-3.2-3B-Instruct: train=108,000 test=12,000 -> datasets/processed_with_performance/math10_categoriesbalanced/Llama-3.2-3B-Instruct_quantized_dataset
